The algotithm ensures that each group first gets one high RL experience participant (mandatory) and then uses a greedy algorithm based on coding experience (secondary) to distribute the remaining participants as evenly as possible.

In [9]:
import pandas as pd
import random
from collections import defaultdict
import numpy as np
import gender_guesser.detector as gender

def load_participants(csv_path):
    df = pd.read_csv(csv_path)
    df = df[df['Do you want to take part in the RL challenge?']=='Yes']
    d = gender.Detector()
    df['sex'] = df['Name'].apply(lambda name: d.get_gender(str(name).split()[0]))
    return df

def compute_diversity_score(group):
    """
    Compute a simple diversity score based on counts of affiliation and inferred sex.
    Higher score means more diversity.
    """
    if len(group) <= 1:
        return 0
    affiliations = group['Affiliation'].value_counts(normalize=True)
    sexes = group['sex'].value_counts(normalize=True)
    affiliation_entropy = -sum(p * np.log(p) for p in affiliations)
    sex_entropy = -sum(p * np.log(p) for p in sexes)
    return affiliation_entropy + sex_entropy

def assign_groups(df, n_groups):
    # Separate participants based on RL experience.
    high_rl = df[df['Rate your RL experience (On an increasing scale of 1-5)'] >= 4]
    low_rl = df[df['Rate your RL experience (On an increasing scale of 1-5)'] < 4]

    if len(high_rl) < n_groups:
        raise ValueError("Not enough high-experience participants for number of groups")

    # Shuffle to randomize assignments.
    high_rl = high_rl.sample(frac=1, random_state=42).reset_index(drop=True)
    low_rl = low_rl.sample(frac=1, random_state=42).reset_index(drop=True)

    groups = defaultdict(list)
    coding_sums = [0] * n_groups  # Track total coding experience per group

    # Start each group with one high RL participant.
    for i in range(n_groups):
        participant = high_rl.iloc[i].to_dict()
        groups[i].append(participant)
        coding_sums[i] += participant['Rate your coding experience (On an increasing scale of 1-5)']

    # Remaining participants: rest of high RL and all low RL.
    remaining_high_rl = high_rl.iloc[n_groups:].to_dict(orient='records')
    remaining_low_rl = low_rl.to_dict(orient='records')
    remaining_participants = remaining_high_rl + remaining_low_rl

    # Sort remaining participants by coding experience descending for balanced distribution.
    remaining_participants.sort(key=lambda x: x['Rate your coding experience (On an increasing scale of 1-5)'], reverse=True)

    # Greedy assignment to the group with the lowest total coding experience.
    for participant in remaining_participants:
        min_group = min(range(n_groups), key=lambda i: coding_sums[i])
        groups[min_group].append(participant)
        coding_sums[min_group] += participant['Rate your coding experience (On an increasing scale of 1-5)']

    return groups

def print_groups(groups):
    for group_id, members in groups.items():
        print(f"\nGroup {group_id + 1}")
        df = pd.DataFrame(members)
        print(df.shape)
        # print(df[['Name',
        #           'Rate your RL experience (On an increasing scale of 1-5)',
        #           'Rate your coding experience (On an increasing scale of 1-5)',
        #           'Affiliation',
        #           'sex']])


In [10]:
csv_path = "registrations_challenge.csv"  # Replace with your CSV path
n_groups = 5                   # Replace with desired number of groups

df = load_participants(csv_path)
groups = assign_groups(df, n_groups)
print_groups(groups)



Group 1
(11, 9)

Group 2
(10, 9)

Group 3
(10, 9)

Group 4
(10, 9)

Group 5
(10, 9)


In [11]:

# Export groups to CSV with a different name
all_participants = []
for group_id, members in groups.items():
    for participant in members:
        participant['group'] = group_id + 1  # Assign group number (starting from 1)
        all_participants.append(participant)
result_df = pd.DataFrame(all_participants)
result_df.to_csv('grouped_registrations_v2.csv', index=False)
print('Exported grouped_registrations_v2.csv')


Exported grouped_registrations_v2.csv
